![RNN](imgs/RNN.png)

## RNN(Recurrent Neural Network) 구현하기

$h_t = f(W_xX_t + W_hh_{t-1} + b)$

$f$는 활성화함수

In [ ]:
import numpy as np

timesteps = 10 # 시점의 수. NLP에서는 보통 문장의 길이가 된다.
input_size = 4 # 입력의 차원. NLP에서는 보통 단어 벡터의 차원이 된다.
hidden_size = 8 # 은닉 상태의 크기. 메모리 셀의 용량이다. 위 그림에서는 h_t의 차원.

inputs = np.random.random((timesteps, input_size)) # 입력에 해당되는 2D 텐서

hidden_state_t = np.zeros((hidden_size,)) # 초기 은닉 상태는 0(벡터)로 초기화
# 은닉 상태의 크기 hidden_size로 은닉 상태를 만듬.


가중치

In [3]:
Wx = np.random.random((hidden_size, input_size))  # (8, 4)크기의 2D 텐서 생성. 입력에 대한 가중치.
Wh = np.random.random((hidden_size, hidden_size)) # (8, 8)크기의 2D 텐서 생성. 은닉 상태에 대한 가중치.
b = np.random.random((hidden_size,)) # (8,)크기의 1D 텐서 생성. 이 값은 편향(bias).

print(np.shape(Wx))
print(np.shape(Wh))
print(np.shape(b))

(8, 4)
(8, 8)
(8,)


In [4]:
total_hidden_states = []

# 메모리 셀 동작
for input_t in inputs: # 각 시점에 따라서 입력값이 입력됨.
    output_t = np.tanh(np.dot(Wx,input_t) + np.dot(Wh,hidden_state_t) + b) # Wx * Xt + Wh * Ht-1 + b(bias)
    total_hidden_states.append(list(output_t)) # 각 시점의 은닉 상태의 값을 계속해서 축적
    print(np.shape(total_hidden_states)) # 각 시점 t별 메모리 셀의 출력의 크기는 (timestep, output_dim)
    hidden_state_t = output_t

total_hidden_states = np.stack(total_hidden_states, axis = 0) 
# 출력 시 값을 깔끔하게 해준다.

print(total_hidden_states) # (timesteps, output_dim)의 크기. 이 경우 (10, 8)의 크기를 가지는 메모리 셀의 2D 텐서를 출력.


(1, 8)
(2, 8)
(3, 8)
(4, 8)
(5, 8)
(6, 8)
(7, 8)
(8, 8)
(9, 8)
(10, 8)
[[0.9839598  0.95456548 0.73214864 0.94412814 0.91079559 0.96469656
  0.98838447 0.94534183]
 [0.99999809 0.99920293 0.99973324 0.99998468 0.99994063 0.99999148
  0.9999955  0.99998072]
 [0.99999885 0.99942022 0.99981689 0.99999099 0.99997114 0.99999559
  0.9999975  0.99997934]
 [0.99999956 0.99969935 0.99989286 0.9999954  0.99997944 0.99999776
  0.9999991  0.99998991]
 [0.99999834 0.99965934 0.99986281 0.99999298 0.99995603 0.99999472
  0.99999761 0.9999781 ]
 [0.99999774 0.99906883 0.99979734 0.99998989 0.99996339 0.99999157
  0.99999687 0.99995532]
 [0.99999893 0.99969878 0.9998324  0.99999234 0.99998122 0.99999759
  0.99999874 0.99997577]
 [0.99999969 0.99980395 0.99990206 0.99999611 0.99998674 0.99999875
  0.99999951 0.99999097]
 [0.99999817 0.99932546 0.99981422 0.99999009 0.99995387 0.99999267
  0.99999593 0.99997563]
 [0.99999643 0.99880254 0.99968856 0.99998172 0.99994878 0.99998902
  0.9999911  0.99995714]

nn.RNN 사용

In [ ]:
import torch, torch.nn as nn
input_size = 5 # 입력의 크기
hidden_size = 8 # 은닉 상태의 크기

# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)

cell = nn.RNN(input_size, hidden_size, batch_first=True) # batch_first = True : 입력 텐서의 첫번째 차원이 배치라는 것을 알려줌.
outputs, _status = cell(inputs)
print(outputs.shape)

torch.Size([1, 10, 8])


Deep RNN

![Deep RNN](imgs/DeepRNN.png)

In [6]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)
cell = nn.RNN(input_size = 5, hidden_size = 8, num_layers = 2, batch_first=True)
outputs, _status = cell(inputs)
print(outputs.shape) # 모든 time-step의 hidden_state
print(_status.shape) # (층의 개수, 배치 크기, 은닉 상태의 크기)

torch.Size([1, 10, 8])
torch.Size([2, 1, 8])


### 양방향 순환 신경망(Bidirectional Recurrent Neural Network)

Exercise is very effective at [    ?      ] belly fat.

1) reducing
2) increasing
3) multiplying

위 경우처럼 과거 시점의 데이터만 고려하는 것이 아니라 향후 시점의 데이터에 힌트가 있는 경우도 많다. 

그래서 이전 시점의 데이터뿐만 아니라, 이후 시점의 데이터도 힌트로 활용하기 위해서 고안된 것이 양방향 RNN이다.

![Bidirectional](imgs/Bidirectional_RNN.png)

양방향 RNN은 하나의 출력값을 예측하기 위해 기본적으로 두 개의 메모리 셀을 사용한다. 

첫번째 메모리 셀은 앞 시점의 은닉 상태(Forward States)를 전달받아 현재의 은닉 상태를 계산한다. (주황색)

두번째 메모리 셀은 뒤 시점의 은닉 상태(Backward States)를 전달 받아 현재의 은닉 상태를 계산한다. (초록색)

그리고 이 두 개의 값 모두가 출력층에서 출력값을 예측하기 위해 사용된다.

In [7]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)
cell = nn.RNN(input_size = 5, hidden_size = 8, num_layers = 2, batch_first=True, bidirectional = True)
outputs, _status = cell(inputs)
print(outputs.shape) # (배치 크기, 시퀀스 길이, 은닉 상태의 크기 x 2)
print(_status.shape) # (층의 개수 x 2, 배치 크기, 은닉 상태의 크기)

torch.Size([1, 10, 16])
torch.Size([4, 1, 8])


### RNN Forward, Backward

![RNNForward](imgs/RNNForward.png)

![RNNBackward](imgs/RNNBackward.gif)

### Vanilla RNN의 한계

![VanillaRNNLimit](imgs/RNNlimit.png)

위의 그림은 첫번째 입력값인 $x_1$의 정보량을 짙은 남색으로 표현했을 때, 

시간이 지나면서 색이 점차 얕아지는 것으로

정보량이 손실되어가는 과정을 표현한 그림이다.

Vanilla RNN의 시점(time step)이 길어질수록 앞의 정보가 뒤로 충분히 전달되지 못하는 현상이 발생한다.

예를 들어 아래 문장의 경우,

    모스크바에 여행을 왔는데 건물도 예쁘고 먹을 것도 맛있었어. 그런데 글쎄 직장 상사한테 전화가 왔어. 어디냐고 묻더라구. 그래서 나는 말했지. 저 여행왔는데요. 여기 ___

장소 정보에 해당되는 단어인 '모스크바'는 앞에 위치하고 있고,

RNN이 충분한 기억력을 가지고 있지 못한다면 다음 단어를 엉뚱하게 예측할 수 있는데,

이를 **장기 의존성 문제(the problem of Long-Term Dependencies)** 라고 한다.

이 문제를 해결하기 위해 등장한 것이 **LSTM** 이다.

[LSTM](lstm.ipynb)

## RNN 실습코드

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# =========================
# 1. 데이터 준비
# =========================
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

char_set = sorted(list(set(sentence)))
char_dic = {c: i for i, c in enumerate(char_set)}
dic_size = len(char_dic)

print("문자 사전:", char_dic)
print("문자 개수:", dic_size)

# 입력 문장과 정답 문장
# 입력:  if you want ...
# 정답:  f you want ...
x_data = [char_dic[c] for c in sentence[:-1]]
y_data = [char_dic[c] for c in sentence[1:]]

# one-hot encoding
x_one_hot = np.eye(dic_size)[x_data]

# batch 차원 추가
X = torch.FloatTensor(x_one_hot).unsqueeze(0)  # (1, seq_len, dic_size)
Y = torch.LongTensor(y_data).unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# =========================
# 2. 모델 정의
# =========================
class Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, layers):
        super(Net, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

# =========================
# 3. 모델 생성
# =========================
input_dim = dic_size
hidden_dim = dic_size
layers = 2

model = Net(input_dim, hidden_dim, layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# =========================
# 4. 학습
# =========================
for epoch in range(300):
    optimizer.zero_grad()

    outputs = model(X)  # (1, seq_len, dic_size)

    loss = criterion(
        outputs.reshape(-1, dic_size),  # (seq_len, dic_size)
        Y.reshape(-1)                   # (seq_len)
    )

    loss.backward()
    optimizer.step()

    if epoch % 30 == 0:
        predicted = outputs.argmax(dim=2).squeeze(0)
        predicted_str = ''.join([char_set[i] for i in predicted])

        print(f"[epoch {epoch:03d}] loss: {loss.item():.4f}")
        print(predicted_str)
        print()

# =========================
# 5. 최종 예측 결과
# =========================
with torch.no_grad():
    outputs = model(X)
    predicted = outputs.argmax(dim=2).squeeze(0)
    predicted_str = ''.join([char_set[i] for i in predicted])

print("입력 문장:")
print(sentence[:-1])

print("\n정답 문장:")
print(sentence[1:])

print("\n모델 예측:")
print(predicted_str)

문자 사전: {' ': 0, "'": 1, ',': 2, '.': 3, 'a': 4, 'b': 5, 'c': 6, 'd': 7, 'e': 8, 'f': 9, 'g': 10, 'h': 11, 'i': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'r': 19, 's': 20, 't': 21, 'u': 22, 'w': 23, 'y': 24}
문자 개수: 25
X shape: torch.Size([1, 179, 25])
Y shape: torch.Size([1, 179])
[epoch 000] loss: 3.2744
rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrirrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrirrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrirrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr

[epoch 030] loss: 1.4180
  tooltond to loild t dhipt don't eost thtpeonla to lthe  to lo la t eo d apd don't eosilm the  to ks ahd woukt dot a the  toe t the  to lo e  orlthe tht dos iod ss  a on the sotp

[epoch 060] loss: 0.2407
f you want to build a ship, don't drum up people together to co lect wood and don't assign them tasks and work, but rather teach them to long for the end ess immensity of the sea.

[epoch 090] loss: 0.0437
f you want to build a ship, don't drum up people together to